In [11]:
import pandas as pd
import sqlite3

In [12]:
df=pd.read_csv("C:/Users/Admin/.cache/kagglehub/datasets/yashch05/direct-to-consumer-e-commerce-funnel-dataset/versions/1/d2c_marketing_funnel_data.csv")

In [13]:
df.head()

,user_id,session_id,date,month,channel,campaign_type,device,user_type,region,visited_website,viewed_product,added_to_cart,checkout_started,purchase_completed,discount_applied,order_value,revenue
0,221958,1,8/16/2025,2025-08,Organic,New Launch,Mobile,New,Metro,Yes,No,No,No,No,No,499.00,0.000
1,771155,2,12/16/2025,2025-12,Organic,Influencer,Mobile,New,Non-Metro,Yes,Yes,Yes,No,No,No,499.00,0.000
2,231932,3,7/17/2025,2025-07,Organic,Influencer,Mobile,New,Non-Metro,Yes,Yes,No,No,No,No,499.00,0.000
3,465838,4,7/4/2025,2025-07,Paid Ads,Discount,Mobile,Returning,Metro,Yes,Yes,Yes,Yes,Yes,Yes,2000.95,1800.855
4,359178,5,8/10/2025,2025-08,Paid Ads,Influencer,Mobile,Returning,Non-Metro,Yes,No,No,No,No,No,499.00,0.000


In [14]:
conn = sqlite3.connect("ecommerce.db")

In [15]:
df.to_sql("ecommerce_data", conn, if_exists="replace", index=False)

120000

In [17]:
pd.read_sql(
"""
SELECT *
FROM ecommerce_data
LIMIT 5
""",
conn)


,user_id,session_id,date,month,channel,campaign_type,device,user_type,region,visited_website,viewed_product,added_to_cart,checkout_started,purchase_completed,discount_applied,order_value,revenue
0,221958,1,8/16/2025,2025-08,Organic,New Launch,Mobile,New,Metro,Yes,No,No,No,No,No,499.00,0.000
1,771155,2,12/16/2025,2025-12,Organic,Influencer,Mobile,New,Non-Metro,Yes,Yes,Yes,No,No,No,499.00,0.000
2,231932,3,7/17/2025,2025-07,Organic,Influencer,Mobile,New,Non-Metro,Yes,Yes,No,No,No,No,499.00,0.000
3,465838,4,7/4/2025,2025-07,Paid Ads,Discount,Mobile,Returning,Metro,Yes,Yes,Yes,Yes,Yes,Yes,2000.95,1800.855
4,359178,5,8/10/2025,2025-08,Paid Ads,Influencer,Mobile,Returning,Non-Metro,Yes,No,No,No,No,No,499.00,0.000


In [ ]:
1. Общая информация о данных
-- Количество пользователей и сессий

In [35]:
query = """
SELECT
COUNT(DISTINCT user_id) AS users,
COUNT(DISTINCT session_id) AS sessions
FROM ecommerce_data
"""

pd.read_sql(query, conn)

,users,sessions
0,112352,120000


In [ ]:
- 2. Основные продуктовые метрики
-- Пользователи, покупки, выручка

In [28]:
query = """
SELECT
COUNT(DISTINCT user_id) AS users,

COUNT(
DISTINCT CASE
WHEN purchase_completed = 'Yes'
THEN user_id END
) AS buyers,

SUM(revenue) AS total_revenue,

AVG(
CASE
WHEN purchase_completed = 'Yes'
THEN revenue
END
) AS average_order_value

FROM ecommerce_data
"""

pd.read_sql(query, conn)

,users,buyers,total_revenue,average_order_value
0,112352,8139,1.701660e+07,2080.014565


In [ ]:
-- 3. Конверсия воронки
-- Сколько пользователей дошли до каждого этапа

In [29]:
query = """
SELECT
'Visited website' AS stage,
COUNT(DISTINCT user_id) AS users
FROM ecommerce_data
WHERE visited_website='Yes'

UNION ALL

SELECT
'Viewed product',
COUNT(DISTINCT user_id)
FROM ecommerce_data
WHERE viewed_product='Yes'

UNION ALL

SELECT
'Added to cart',
COUNT(DISTINCT user_id)
FROM ecommerce_data
WHERE added_to_cart='Yes'

UNION ALL

SELECT
'Checkout started',
COUNT(DISTINCT user_id)
FROM ecommerce_data
WHERE checkout_started='Yes'

UNION ALL

SELECT
'Purchase completed',
COUNT(DISTINCT user_id)
FROM ecommerce_data
WHERE purchase_completed='Yes'
"""

pd.read_sql(query, conn)

,stage,users
0,Visited website,112352
1,Viewed product,74579
2,Added to cart,26709
3,Checkout started,16089
4,Purchase completed,8139


In [ ]:
-- 4. Анализ каналов привлечения
-- Какой канал приносит больше покупателей

In [30]:
query = """
SELECT

channel,

COUNT(DISTINCT user_id) AS users,

COUNT(
DISTINCT CASE
WHEN purchase_completed='Yes'
THEN user_id END
) AS buyers,

SUM(revenue) AS revenue,

SUM(revenue) /
COUNT(DISTINCT user_id)
AS revenue_per_user

FROM ecommerce_data

GROUP BY channel

ORDER BY revenue DESC
"""

pd.read_sql(query, conn)

,channel,users,buyers,revenue,revenue_per_user
0,Paid Ads,52346,3612,7536147.080,143.967965
1,Organic,35225,2447,5090708.447,144.519757
2,Social,17897,1228,2578443.312,144.071258
3,Email,12019,884,1811300.315,150.703080


In [ ]:
-- 5. Влияние скидки
-- Сравнение пользователей со скидкой и без

In [31]:
query = """
SELECT

discount_applied,

COUNT(DISTINCT user_id) AS users,

COUNT(
CASE WHEN purchase_completed='Yes'
THEN user_id END
)
AS buyers,

AVG(
CASE WHEN purchase_completed='Yes'
THEN revenue END
)
AS avg_order_value

FROM ecommerce_data

GROUP BY discount_applied
"""

pd.read_sql(query, conn)

,discount_applied,users,buyers,avg_order_value
0,No,108459,3719,2200.976628
1,Yes,4447,4462,1979.194772


In [ ]:
-- 6. Поведение новых и возвращающихся пользователей

In [32]:
query = """
SELECT

user_type,

COUNT(DISTINCT user_id) AS users,

COUNT(
CASE WHEN purchase_completed='Yes'
THEN user_id END
)
AS buyers,

AVG(revenue) AS avg_revenue

FROM ecommerce_data

GROUP BY user_type
"""

pd.read_sql(query, conn)

,user_type,users,buyers,avg_revenue
0,New,74723,5398,143.977759
1,Returning,41062,2783,137.774435


In [ ]:
-- 7. Анализ устройств
-- Есть ли разница Mobile/Desktop

In [33]:
query = """
SELECT

device,

COUNT(DISTINCT user_id) AS users,

COUNT(
CASE WHEN purchase_completed='Yes'
THEN user_id END
)
AS buyers,

SUM(revenue) AS revenue

FROM ecommerce_data

GROUP BY device
"""

pd.read_sql(query, conn)

,device,users,buyers,revenue
0,Desktop,35285,2488,5.172967e+06
1,Mobile,80179,5693,1.184363e+07


In [ ]:
-- 8. Анализ регионов
-- Где выше активность

In [34]:
query = """
SELECT

region,

COUNT(DISTINCT user_id) AS users,

SUM(revenue) AS revenue

FROM ecommerce_data

GROUP BY region

ORDER BY revenue DESC
"""

pd.read_sql(query, conn)

,region,users,revenue
0,Metro,69230,1.026102e+07
1,Non-Metro,46739,6.755580e+06
